# 02 — Fine-tune DistilBERT on Colab or Kaggle

This notebook runs `src/train_transformer.py`. The model is the pretrained checkpoint `distilbert-base-uncased`. Nothing is trained from scratch.

Use a GPU runtime. A T4 is enough for two epochs at batch size 16 and sequence length 256. The script turns on fp16 only when CUDA is visible.

## Colab
1. Runtime → Change runtime type → GPU.
2. Get the project onto the machine with the setup cell (git clone or zip upload). Include `data/processed/train.csv`, `val.csv`, and `test.csv`. Those files are produced locally by `python -m src.data_utils`.
3. Run the install cell, confirm the GPU check prints a device name, then run training.

## Kaggle
1. Settings → Accelerator → GPU. Turn Internet on so pip can install the pinned libraries and download DistilBERT.
2. `/kaggle/input` is read-only. Attach the project as a dataset, copy it into `/kaggle/working`, and train there. Checkpoints and metrics cannot be written under `/kaggle/input`.
3. When the run finishes, download `models/distilbert_detector/` and `outputs/` from the working directory before the session ends.


## 1. Put the project on this machine

Leave the next cell as it is if you opened this notebook from a copy of `ai-text-detector` that is already on the machine. Otherwise uncomment **one** block, edit the placeholder, and run the cell.


In [ ]:
# --- Colab or any host with git ---
# !git clone https://github.com/YOUR_USER/YOUR_REPO.git ai-text-detector
# %cd ai-text-detector

# --- Colab zip upload ---
# Zip the project first, including data/processed/*.csv.
# from google.colab import files
# files.upload()  # choose ai-text-detector.zip
# !unzip -q -o ai-text-detector.zip
# %cd ai-text-detector

# --- Kaggle ---
# Attach the project as a Kaggle dataset, then copy it out of read-only input.
# !cp -a /kaggle/input/YOUR_DATASET/ai-text-detector /kaggle/working/ai-text-detector
# %cd /kaggle/working/ai-text-detector

print("Setup cell ready. Uncomment one block above if the project is not already in the working directory.")


In [ ]:
import os
import sys
from pathlib import Path

def find_project_root() -> Path:
    start = Path.cwd().resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "src" / "train_transformer.py").is_file():
            return candidate
    if start.is_dir():
        for child in start.iterdir():
            if child.is_dir() and (child / "src" / "train_transformer.py").is_file():
                return child
    raise FileNotFoundError(
        "Could not find src/train_transformer.py. "
        "Uncomment a clone, zip, or Kaggle copy block in the setup cell and run it first."
    )

ROOT = find_project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)


## 2. Install dependencies

Colab and Kaggle already have a CUDA build of PyTorch. `requirements.txt` pins `torch==2.14.1` and the rest of the course stack. The Linux wheel from PyPI is the CUDA build, so do not add a CPU-only `--extra-index-url`.

If the install replaces the preinstalled PyTorch, use Runtime → Restart session on Colab (or restart the Kaggle kernel), then rerun from the project-root cell. The GPU check below must print `cuda available: True` before a full training run.


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
print("Installed requirements with", sys.executable)


## 3. Check the GPU

`src/train_transformer.py` logs the same fact: fp16 is on only when `torch.cuda.is_available()` is true. A CPU session still runs, but the full corpus will be very slow. Use `--max_samples 500` for that smoke test.


In [ ]:
import torch

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("fp16 will be enabled by the training script.")
else:
    print("No GPU visible.")
    print("Colab: Runtime -> Change runtime type -> GPU, then restart and rerun.")
    print("Kaggle: Settings -> Accelerator -> GPU, then restart and rerun.")
    print("Until a GPU is visible, set smoke_test = True in the training cell.")


## 4. Confirm the processed splits

Training reads `data/processed/train.csv`, `val.csv`, and `test.csv`. Build them on your own machine with `python -m src.data_utils` and include them in the zip or clone. The raw Kaggle CSV is only needed if you want to rebuild the splits here.


In [ ]:
from pathlib import Path

processed = Path("data/processed")
required = ["train.csv", "val.csv", "test.csv"]
missing = [name for name in required if not (processed / name).is_file()]
if missing:
    raise FileNotFoundError(
        "Missing "
        + ", ".join(f"data/processed/{name}" for name in missing)
        + ". Include those files in the upload, or place data/raw/AI_Human.csv "
        + "and run: python -m src.data_utils"
    )
for name in required:
    path = processed / name
    print(f"{path.as_posix()}  {path.stat().st_size / 1e6:.1f} MB")


## 5. Train

`smoke_test = False` fine-tunes on every processed row for the epoch count, batch size, and learning rate in `src/config.py` (2 epochs, batch size 16, learning rate `2e-5`).

Set `smoke_test = True` before a first run, or whenever CUDA is unavailable. That passes `--max_samples 500`, which keeps at most 500 stratified rows in each split.

If the GPU runs out of memory, stop the run, lower `BATCH_SIZE` in `src/config.py`, and start again. The script deletes `models/distilbert_checkpoints` at the start of each run so an old checkpoint cannot be reloaded by mistake.


In [ ]:
import subprocess
import sys

# True: quick check with --max_samples 500. False: full fine-tune.
smoke_test = False

command = [sys.executable, "-m", "src.train_transformer"]
if smoke_test:
    command.extend(["--max_samples", "500"])

print("Running:", " ".join(command))
subprocess.run(command, check=True)


## 6. Where the artifacts are

- `models/distilbert_detector/` — best DistilBERT weights and tokenizer, chosen by validation F1
- `outputs/metrics/transformer_metrics.json` — test accuracy, precision, recall, and F1
- `outputs/metrics/transformer_test_predictions.csv` — test text, gold label, prediction, and class probabilities
- `outputs/figures/cm_transformer.png` — test confusion matrix
- `outputs/figures/loss_curve.png` — training and validation loss by epoch

On Colab, use the file browser to download those paths. On Kaggle, download them from `/kaggle/working` before the session is discarded. Intermediate epoch checkpoints live in `models/distilbert_checkpoints/` and are not the graded model; the reloaded best weights are in `models/distilbert_detector/`.
